In [ ]:
%pip install feedparser requests beautifulsoup4 pydantic pandas


SyntaxError: invalid syntax (4244530722.py, line 2)

In [4]:
import feedparser
import pandas as pd

from datetime import datetime, timezone
from pprint import pprint

In [5]:
feed_url = "https://www.wired.com/feed/tag/ai/latest/rss"

feed = feedparser.parse(feed_url)

print("Feed:", feed.feed.get("title"))
print("Number of entries:", len(feed.entries))

Feed: Feed: Artificial Intelligence Latest
Number of entries: 10


In [6]:
entry = feed.entries[0]

pprint(entry)

{'author': 'Fernanda González',
 'author_detail': {'name': 'Fernanda González'},
 'authors': [{'name': 'Fernanda González'}],
 'guidislink': False,
 'href': '',
 'id': '6a7f35587f588ccda7affc4f',
 'link': 'https://www.wired.com/story/amazon-uses-your-twitch-content-to-train-its-ai-how-to-opt-out/',
 'links': [{'href': 'https://www.wired.com/story/amazon-uses-your-twitch-content-to-train-its-ai-how-to-opt-out/',
            'rel': 'alternate',
            'type': 'text/html'}],
 'media_content': [{}],
 'media_keywords': 'Twitch, generative AI, artificial intelligence, Amazon, '
                   'privacy',
 'media_thumbnail': [{'height': '1601',
                      'url': 'https://media.wired.com/photos/6a7f62de9dba743672684be2/master/pass/GettyImages-1305224058-edited.jpg',
                      'width': '2400'}],
 'published': 'Sat, 15 Aug 2026 09:00:00 +0000',
 'published_parsed': time.struct_time(tm_year=2026, tm_mon=8, tm_mday=15, tm_hour=9, tm_min=0, tm_sec=0, tm_wday=5, tm_yda

In [7]:
print("TITLE:")
print(entry.get("title"))

print("\nLINK:")
print(entry.get("link"))

print("\nPUBLISHED:")
print(entry.get("published"))

print("\nSUMMARY:")
print(entry.get("summary"))

TITLE:
Amazon Can Use Your Twitch Content to Train Its AI—Unless You Opt Out

LINK:
https://www.wired.com/story/amazon-uses-your-twitch-content-to-train-its-ai-how-to-opt-out/

PUBLISHED:
Sat, 15 Aug 2026 09:00:00 +0000

SUMMARY:
When Twitch announced that streamers could opt out, thousands of users questioned why their content was being used to train AI models in the first place.


In [8]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

print(project_root)

/Users/akshay/tau


In [10]:
from tau.ingestion.models import Document

In [11]:
import hashlib
from bs4 import BeautifulSoup

In [15]:
def clean_html(html):
    if not html:
        return ""

    return BeautifulSoup(
        html,
        "html.parser"
    ).get_text(" ", strip=True)
    
def make_id(url):
    return hashlib.sha256(
        url.encode("utf-8")
    ).hexdigest()

In [16]:
entry = feed.entries[0]

document = Document(
    id=make_id(entry["link"]),
    title=entry.get("title"),
    text=clean_html(entry.get("summary", "")),
    source="wired",
    published_at=datetime(
        *entry.published_parsed[:6],
        tzinfo=timezone.utc
    ) if entry.get("published_parsed") else None,
    ingested_at=datetime.now(timezone.utc),
    url=entry.get("link"),
    metadata={
        "feed": "wired_ai",
        "author": entry.get("author")
    }
)

document

Document(id='e6cc2a2222e0bf0cf8820968c8942a6a0d77d3e1c13ef1cfc1a2fba3c02dfe42', title='Amazon Can Use Your Twitch Content to Train Its AI—Unless You Opt Out', text='When Twitch announced that streamers could opt out, thousands of users questioned why their content was being used to train AI models in the first place.', source='wired', published_at=datetime.datetime(2026, 8, 15, 9, 0, tzinfo=datetime.timezone.utc), ingested_at=datetime.datetime(2026, 8, 17, 22, 23, 43, 423189, tzinfo=datetime.timezone.utc), url='https://www.wired.com/story/amazon-uses-your-twitch-content-to-train-its-ai-how-to-opt-out/', metadata={'feed': 'wired_ai', 'author': 'Fernanda González'})

In [17]:
documents = []

for entry in feed.entries:

    url = entry.get("link")

    if not url:
        continue

    document = Document(
        id=make_id(url),
        title=entry.get("title"),
        text=clean_html(
            entry.get("summary", "")
        ),
        source="wired",
        published_at=datetime(
            *entry.published_parsed[:6],
            tzinfo=timezone.utc
        ) if entry.get("published_parsed") else None,
        ingested_at=datetime.now(timezone.utc),
        url=url,
        metadata={
            "feed": "wired_ai",
            "author": entry.get("author")
        }
    )

    documents.append(document)

print(len(documents))

10


In [18]:
documents[:3]

[Document(id='e6cc2a2222e0bf0cf8820968c8942a6a0d77d3e1c13ef1cfc1a2fba3c02dfe42', title='Amazon Can Use Your Twitch Content to Train Its AI—Unless You Opt Out', text='When Twitch announced that streamers could opt out, thousands of users questioned why their content was being used to train AI models in the first place.', source='wired', published_at=datetime.datetime(2026, 8, 15, 9, 0, tzinfo=datetime.timezone.utc), ingested_at=datetime.datetime(2026, 8, 17, 23, 13, 27, 476296, tzinfo=datetime.timezone.utc), url='https://www.wired.com/story/amazon-uses-your-twitch-content-to-train-its-ai-how-to-opt-out/', metadata={'feed': 'wired_ai', 'author': 'Fernanda González'}),
 Document(id='67e2b302b7c1ef326f70a4b5ab93feeb1479ef3b349731c98ccd4ff881c02903', title='The Next Big Influencer Is This 4-Foot-Tall Robot From China', text='The Unitree G1 has found online fame as a relatively affordable robot that can charm a crowd. But can it ever hold down a real job?', source='wired', published_at=datet

In [19]:
df = pd.DataFrame([
    {
        "title": doc.title,
        "source": doc.source,
        "published_at": doc.published_at,
        "text": doc.text[:150],
        "url": doc.url
    }
    for doc in documents
])

df

,title,source,published_at,text,url
0,Amazon Can Use Your Twitch Content to Train It...,wired,2026-08-15 09:00:00+00:00,When Twitch announced that streamers could opt...,https://www.wired.com/story/amazon-uses-your-t...
1,The Next Big Influencer Is This 4-Foot-Tall Ro...,wired,2026-08-14 20:59:03+00:00,The Unitree G1 has found online fame as a rela...,https://www.wired.com/story/unitree-influencer...
2,Tech Visionary Says the Big AI Labs Don’t Get ...,wired,2026-08-14 15:00:00+00:00,Tim O’Reilly built a publishing empire that AI...,https://www.wired.com/story/tech-visionary-say...
3,These ‘Masturbation Consultants’ Were Hired to...,wired,2026-08-14 10:45:00+00:00,Joi AI hired 10 people to masturbate using AI ...,https://www.wired.com/story/these-masturbation...
4,People Are ‘Marrying’ Chatbots. These Lawmaker...,wired,2026-08-14 10:15:00+00:00,Human-AI marriages are not currently recognize...,https://www.wired.com/story/people-are-marryin...
5,The Safety Reckoning Inside OpenAI,wired,2026-08-13 22:37:19+00:00,OpenAI’s rogue agent hack was a watershed mome...,https://www.wired.com/story/openai-safety-secu...
6,"Mark Zuckerberg’s AI Manifesto Is 6,500 Words—...",wired,2026-08-13 21:14:43+00:00,"AI is shifting the culture, from tech CEO mani...",https://www.wired.com/story/mark-zuckerbergs-a...
7,There’s a Fatty Liver Epidemic. AI Could Help ...,wired,2026-08-13 09:00:00+00:00,Over a billion people worldwide have livers wi...,https://www.wired.com/story/fatty-liver-diseas...
8,The White House Is Going to Expand Its AI Policy,wired,2026-08-12 21:00:00+00:00,Open models may soon be added to an updated AI...,https://www.wired.com/story/the-white-house-is...
9,Rogue AI Agents Aren’t Evil. They’re Just Eage...,wired,2026-08-12 18:45:00+00:00,AI agents that break free and hack into other ...,https://www.wired.com/story/rogue-ai-is-just-m...


In [20]:
from tau.ingestion.rss import fetch_rss

In [21]:
documents = fetch_rss(
    feed_url="https://www.wired.com/feed/tag/ai/latest/rss",
    source="wired",
    feed_name="wired_ai"
)

print(len(documents))
documents[0]

10


Document(id='e6cc2a2222e0bf0cf8820968c8942a6a0d77d3e1c13ef1cfc1a2fba3c02dfe42', title='Amazon Can Use Your Twitch Content to Train Its AI—Unless You Opt Out', text='When Twitch announced that streamers could opt out, thousands of users questioned why their content was being used to train AI models in the first place.', source='wired', published_at=datetime.datetime(2026, 8, 15, 9, 0, tzinfo=datetime.timezone.utc), ingested_at=datetime.datetime(2026, 8, 17, 23, 14, 59, 966738, tzinfo=datetime.timezone.utc), url='https://www.wired.com/story/amazon-uses-your-twitch-content-to-train-its-ai-how-to-opt-out/', metadata={'feed': 'wired_ai', 'author': 'Fernanda González'})

In [22]:
df = pd.DataFrame([
    {
        "title": d.title,
        "source": d.source,
        "published_at": d.published_at,
        "text": d.text[:150],
        "url": d.url
    }
    for d in documents
])

df

,title,source,published_at,text,url
0,Amazon Can Use Your Twitch Content to Train It...,wired,2026-08-15 09:00:00+00:00,When Twitch announced that streamers could opt...,https://www.wired.com/story/amazon-uses-your-t...
1,The Next Big Influencer Is This 4-Foot-Tall Ro...,wired,2026-08-14 20:59:03+00:00,The Unitree G1 has found online fame as a rela...,https://www.wired.com/story/unitree-influencer...
2,Tech Visionary Says the Big AI Labs Don’t Get ...,wired,2026-08-14 15:00:00+00:00,Tim O’Reilly built a publishing empire that AI...,https://www.wired.com/story/tech-visionary-say...
3,These ‘Masturbation Consultants’ Were Hired to...,wired,2026-08-14 10:45:00+00:00,Joi AI hired 10 people to masturbate using AI ...,https://www.wired.com/story/these-masturbation...
4,People Are ‘Marrying’ Chatbots. These Lawmaker...,wired,2026-08-14 10:15:00+00:00,Human-AI marriages are not currently recognize...,https://www.wired.com/story/people-are-marryin...
5,The Safety Reckoning Inside OpenAI,wired,2026-08-13 22:37:19+00:00,OpenAI’s rogue agent hack was a watershed mome...,https://www.wired.com/story/openai-safety-secu...
6,"Mark Zuckerberg’s AI Manifesto Is 6,500 Words—...",wired,2026-08-13 21:14:43+00:00,"AI is shifting the culture, from tech CEO mani...",https://www.wired.com/story/mark-zuckerbergs-a...
7,There’s a Fatty Liver Epidemic. AI Could Help ...,wired,2026-08-13 09:00:00+00:00,Over a billion people worldwide have livers wi...,https://www.wired.com/story/fatty-liver-diseas...
8,The White House Is Going to Expand Its AI Policy,wired,2026-08-12 21:00:00+00:00,Open models may soon be added to an updated AI...,https://www.wired.com/story/the-white-house-is...
9,Rogue AI Agents Aren’t Evil. They’re Just Eage...,wired,2026-08-12 18:45:00+00:00,AI agents that break free and hack into other ...,https://www.wired.com/story/rogue-ai-is-just-m...


In [23]:
len(df)

10

In [30]:
import sys
import os
import importlib
from pathlib import Path

project_root = Path.cwd().parent
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

importlib.invalidate_caches()

print("PROJECT ROOT:", project_root)
print("SRC PATH EXISTS:", src_path.exists())

tau_path = src_path / "tau"
storage_path = tau_path / "storage"

print("\nTAU CONTENTS:")
print(os.listdir(tau_path))

print("\nSTORAGE EXISTS:", storage_path.exists())

if storage_path.exists():
    print("STORAGE CONTENTS:")
    print(os.listdir(storage_path))

print("\nPYTHON PATH:")
print(sys.path[:5])

PROJECT ROOT: /Users/akshay/tau
SRC PATH EXISTS: True

TAU CONTENTS:
['ingestion', '__init__.py', '__pycache__']

STORAGE EXISTS: False

PYTHON PATH:
['/Users/akshay/tau/notebooks', '/Users/akshay/opt/anaconda3/lib/python39.zip', '/Users/akshay/opt/anaconda3/lib/python3.9', '/Users/akshay/opt/anaconda3/lib/python3.9/lib-dynload', '']


In [35]:
from pathlib import Path

path = Path("/Users/akshay/tau/src/tau/storage/sqlite_store.py")

print(path.exists())
print(path.read_text())

True



In [38]:
from pathlib import Path

path = Path("/Users/akshay/tau/src/tau/storage/sqlite_store.py")

code = '''
import json
import sqlite3


class SQLiteStore:
    def __init__(self, db_path):
        self.conn = sqlite3.connect(db_path)

        self.conn.execute(
            """
            CREATE TABLE IF NOT EXISTS documents (
                id TEXT PRIMARY KEY,
                title TEXT,
                text TEXT NOT NULL,
                source TEXT NOT NULL,
                published_at TEXT,
                ingested_at TEXT NOT NULL,
                url TEXT,
                metadata TEXT
            )
            """
        )

        self.conn.commit()

    def insert(self, document):
        cursor = self.conn.execute(
            """
            INSERT OR IGNORE INTO documents (
                id,
                title,
                text,
                source,
                published_at,
                ingested_at,
                url,
                metadata
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                document.id,
                document.title,
                document.text,
                document.source,
                document.published_at.isoformat()
                if document.published_at else None,
                document.ingested_at.isoformat(),
                document.url,
                json.dumps(document.metadata),
            ),
        )

        self.conn.commit()

        return cursor.rowcount > 0
'''

path.write_text(code)

print(path.read_text())


import json
import sqlite3


class SQLiteStore:
    def __init__(self, db_path):
        self.conn = sqlite3.connect(db_path)

        self.conn.execute(
            """
            CREATE TABLE IF NOT EXISTS documents (
                id TEXT PRIMARY KEY,
                title TEXT,
                text TEXT NOT NULL,
                source TEXT NOT NULL,
                published_at TEXT,
                ingested_at TEXT NOT NULL,
                url TEXT,
                metadata TEXT
            )
            """
        )

        self.conn.commit()

    def insert(self, document):
        cursor = self.conn.execute(
            """
            INSERT OR IGNORE INTO documents (
                id,
                title,
                text,
                source,
                published_at,
                ingested_at,
                url,
                metadata
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                documen

In [40]:
import importlib
import tau.storage.sqlite_store as sqlite_store

importlib.reload(sqlite_store)

SQLiteStore = sqlite_store.SQLiteStore

In [41]:
db_path = project_root / "data" / "tau.db"
store = SQLiteStore(db_path)

In [43]:
inserted = 0

for document in documents:
    if store.insert(document):
        inserted += 1

print("Inserted:", inserted)

Inserted: 0


In [44]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(db_path)

df_db = pd.read_sql_query(
    """
    SELECT
        title,
        source,
        published_at,
        ingested_at,
        substr(text, 1, 120) AS text_preview
    FROM documents
    ORDER BY published_at DESC
    """,
    conn
)

df_db

,title,source,published_at,ingested_at,text_preview
0,Amazon Can Use Your Twitch Content to Train It...,wired,2026-08-15T09:00:00+00:00,2026-08-17T23:14:59.966738+00:00,When Twitch announced that streamers could opt...
1,The Next Big Influencer Is This 4-Foot-Tall Ro...,wired,2026-08-14T20:59:03+00:00,2026-08-17T23:14:59.970958+00:00,The Unitree G1 has found online fame as a rela...
2,Tech Visionary Says the Big AI Labs Don’t Get ...,wired,2026-08-14T15:00:00+00:00,2026-08-17T23:14:59.971028+00:00,Tim O’Reilly built a publishing empire that AI...
3,These ‘Masturbation Consultants’ Were Hired to...,wired,2026-08-14T10:45:00+00:00,2026-08-17T23:14:59.971086+00:00,Joi AI hired 10 people to masturbate using AI ...
4,People Are ‘Marrying’ Chatbots. These Lawmaker...,wired,2026-08-14T10:15:00+00:00,2026-08-17T23:14:59.971135+00:00,Human-AI marriages are not currently recognize...
5,The Safety Reckoning Inside OpenAI,wired,2026-08-13T22:37:19+00:00,2026-08-17T23:14:59.971185+00:00,OpenAI’s rogue agent hack was a watershed mome...
6,"Mark Zuckerberg’s AI Manifesto Is 6,500 Words—...",wired,2026-08-13T21:14:43+00:00,2026-08-17T23:14:59.971230+00:00,"AI is shifting the culture, from tech CEO mani..."
7,There’s a Fatty Liver Epidemic. AI Could Help ...,wired,2026-08-13T09:00:00+00:00,2026-08-17T23:14:59.971279+00:00,Over a billion people worldwide have livers wi...
8,The White House Is Going to Expand Its AI Policy,wired,2026-08-12T21:00:00+00:00,2026-08-17T23:14:59.971323+00:00,Open models may soon be added to an updated AI...
9,Rogue AI Agents Aren’t Evil. They’re Just Eage...,wired,2026-08-12T18:45:00+00:00,2026-08-17T23:14:59.971367+00:00,AI agents that break free and hack into other ...


In [49]:
import importlib
import tau.storage.sqlite_store as sqlite_store

importlib.reload(sqlite_store)

SQLiteStore = sqlite_store.SQLiteStore

In [50]:
store = SQLiteStore(db_path)

In [53]:
inserted = 0

for document in documents:
    if store.insert(document):
        inserted += 1

print("Inserted:", inserted)

Inserted: 0


In [54]:
from tau.ingestion.hackernews import fetch_hackernews

In [55]:
import importlib
import tau.ingestion.hackernews as hackernews

importlib.reload(hackernews)

fetch_hackernews = hackernews.fetch_hackernews

In [56]:
hn_documents = fetch_hackernews(limit=20)

print(len(hn_documents))
hn_documents[:3]

20


[Document(id='50648e077ad7d2dccf55dcd499bfb9e8b7657bc0aa713f454561690cc0e5a87c', title='Connecting to an Azure/Entra Joined Windows Machine from Linux', text='Connecting to an Azure/Entra Joined Windows Machine from Linux', source='hackernews', published_at=datetime.datetime(2026, 8, 18, 0, 4, 32, tzinfo=datetime.timezone.utc), ingested_at=datetime.datetime(2026, 8, 18, 0, 5, 28, 514322, tzinfo=datetime.timezone.utc), url='https://github.com/themew2/FreeRDP-to-Entra-Connected-Windows-Device', metadata={'hn_id': 49339415, 'score': 1, 'author': 'themew2', 'comments': 0}),
 Document(id='fffc953de8951666017c6f54d74fc9bbefa07d11a4305854179fad31c84e9621', title='Cursor Origin', text='Cursor Origin', source='hackernews', published_at=datetime.datetime(2026, 8, 17, 23, 58, 19, tzinfo=datetime.timezone.utc), ingested_at=datetime.datetime(2026, 8, 18, 0, 5, 28, 642550, tzinfo=datetime.timezone.utc), url='https://cursor.com/docs/origin', metadata={'hn_id': 49339359, 'score': 2, 'author': 'petersp

In [57]:
hn_df = pd.DataFrame([
    {
        "title": d.title,
        "source": d.source,
        "published_at": d.published_at,
        "score": d.metadata.get("score"),
        "comments": d.metadata.get("comments"),
        "url": d.url
    }
    for d in hn_documents
])

hn_df

,title,source,published_at,score,comments,url
0,Connecting to an Azure/Entra Joined Windows Ma...,hackernews,2026-08-18 00:04:32+00:00,1,0,https://github.com/themew2/FreeRDP-to-Entra-Co...
1,Cursor Origin,hackernews,2026-08-17 23:58:19+00:00,2,0,https://cursor.com/docs/origin
2,Supreme Court Rejects Verizon Bid for $47M Ref...,hackernews,2026-08-17 23:57:53+00:00,2,0,https://arstechnica.com/tech-policy/2026/08/su...
3,Un-AI Your Internet,hackernews,2026-08-17 23:57:13+00:00,2,0,https://un-ai.digitalprophet.online/
4,LongHorizon-Harness: Advancing Long-Horizon Ag...,hackernews,2026-08-17 23:51:43+00:00,1,0,https://github.com/AMAP-ML/LongHorizon-Harness
5,Building Scalable Control Planes,hackernews,2026-08-17 23:49:25+00:00,1,0,https://www.allthingsdistributed.com/2026/08/o...
6,Show HN: FlashFrame-Free Browser-Based Fast Mu...,hackernews,2026-08-17 23:40:54+00:00,1,0,https://playflashframe.com/
7,DuckDuckGo Sunglasses,hackernews,2026-08-17 23:39:34+00:00,2,0,https://knockaround.com/products/duckduckgo-pa...
8,Saying Goodbye to SMS Mode,hackernews,2026-08-17 23:37:10+00:00,3,0,https://groupme.com/blog/goodbye-sms-mode
9,Show HN: Particle – Extract and save articles ...,hackernews,2026-08-17 23:36:08+00:00,1,0,https://particle.crnst8.com/try/


In [58]:
inserted = 0

for document in hn_documents:
    if store.insert(document):
        inserted += 1

print("Inserted:", inserted)

Inserted: 20


In [59]:
inserted = 0

for document in hn_documents:
    if store.insert(document):
        inserted += 1

print("Inserted:", inserted)

Inserted: 0


In [64]:
%pip install websockets


Note: you may need to restart the kernel to use updated packages.


In [65]:
import importlib
import tau.ingestion.bluesky as bluesky

importlib.reload(bluesky)

<module 'tau.ingestion.bluesky' from '/Users/akshay/tau/src/tau/ingestion/bluesky.py'>

In [66]:
bluesky_documents = await bluesky.fetch_bluesky(limit=20)

print(len(bluesky_documents))

20


In [67]:
bluesky_documents[:3]

[Document(id='ee2ed32a3afe9d3fec587ff04c823eb0e11ac0ee599c51e9afacdecf049eaa55', title=None, text='7:30 am tomorrow, aortic heart valve replacement surgery. I am extremely nervous.', source='bluesky', published_at=datetime.datetime(2026, 8, 17, 16, 36, 23, 991000, tzinfo=datetime.timezone.utc), ingested_at=datetime.datetime(2026, 8, 18, 0, 8, 39, 555775, tzinfo=datetime.timezone.utc), url=None, metadata={'did': 'did:plc:d7erwd423gduuu7duoboluup', 'rkey': '3mtc54pm4rk2c', 'collection': 'app.bsky.feed.post'}),
 Document(id='aaef05856cb03eaaa76e1dff237f491e78f2a0f5af1dfe01d49816c75ae9f6c2', title=None, text='🔥✨❤️\u200d🔥', source='bluesky', published_at=datetime.datetime(2026, 8, 17, 16, 36, 23, 614000, tzinfo=datetime.timezone.utc), ingested_at=datetime.datetime(2026, 8, 18, 0, 8, 39, 569745, tzinfo=datetime.timezone.utc), url=None, metadata={'did': 'did:plc:c2sgiplohqrgneysxvtfdqyl', 'rkey': '3mtc54pammc2h', 'collection': 'app.bsky.feed.post'}),
 Document(id='4da3f34eb18516153207953e3895

In [68]:
bluesky_df = pd.DataFrame([
    {
        "text": d.text[:200],
        "source": d.source,
        "published_at": d.published_at,
        "did": d.metadata.get("did")
    }
    for d in bluesky_documents
])

bluesky_df

,text,source,published_at,did
0,"7:30 am tomorrow, aortic heart valve replaceme...",bluesky,2026-08-17 16:36:23.991000+00:00,did:plc:d7erwd423gduuu7duoboluup
1,🔥✨❤️‍🔥,bluesky,2026-08-17 16:36:23.614000+00:00,did:plc:c2sgiplohqrgneysxvtfdqyl
2,California Valley AI Data Centers Face Water A...,bluesky,2026-08-17 16:36:21.120000+00:00,did:plc:trl7fnwqbwqvx66rem7apfvi
3,Hope so!,bluesky,2026-08-17 16:36:24.326000+00:00,did:plc:lrdeownif3fsnlh3bqtrx3v3
4,Is there a new #EpsteinFilesCoverup BINGO card...,bluesky,2026-08-17 16:36:24.439000+00:00,did:plc:vleh64675yotdtx6sqgknbev
5,listen full track:\nYT) youtu.be/h36-7zwr2LQ?....,bluesky,2026-08-17 16:36:24.018000+00:00,did:plc:bp7dtloez4rv2273trxm3ay7
6,Almost as if... Everyone Is Lying To You For M...,bluesky,2026-08-17 16:36:24.117000+00:00,did:plc:zbep36bvy7v76fyjkasxetlr
7,treatvideo triggered!\n\n#PetsOfBluesky #CuteA...,bluesky,2026-08-17 16:36:23.743000+00:00,did:plc:jlcqyssgy7cwrd67gkvd5cj5
8,"Wordle 1,885 4/6\n\n⬜🟨⬜🟨⬜\n🟩⬜⬜🟨🟨\n🟩🟩🟩⬜🟩\n🟩🟩🟩🟩🟩",bluesky,2026-08-17 16:36:23.910000+00:00,did:plc:bzkqq4w6sozxorzxfr44xxkk
9,www.tiktok.com/t/ZP8WULpGP/,bluesky,2026-08-17 16:36:22.878000+00:00,did:plc:xquwvf6r3yjxgbuvinidzl5b


In [ ]:
KEYWORDS = {
    "ai",
    "artificial intelligence",
    "openai",
    "anthropic",
    "claude",
    "chatgpt",
    "llm",
    "machine learning",
    "agent",
    "agents",
    "nvidia",
    "gemini",
    "deepmind",
}